# 🌳 SF Street Tree Inventory - Data Ingest
### Powered by Elasticsearch Serverless + Mistral

San Francisco Public Works keeps an inventory of **~145,000 street trees**: species, size, when it was planted, who looks after it, and exactly where it stands. This notebook loads all of them into Elasticsearch.

**What this notebook does:**
1. Fetches the inventory from [DataSF](https://data.sf.gov/d/tkzw-k3nq) - no API key required
2. Splits species into scientific and common name, parses size and planting date, builds a `geo_point`
3. Bulk-indexes everything into `sf_street_trees`

---
> **Data source:** San Francisco Street Tree Inventory, dataset `tkzw-k3nq`. Refreshed regularly by Public Works.

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests --quiet

In [ ]:
import requests
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless
> **Elastic Cloud Console → Your Project → Connection Details**

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)
print(f'✅ Connected to Elasticsearch {es.info()["version"]["number"]}')

## 2. Fetch the trees
About 145k rows - it takes a minute or two. Lower `SAMPLE_SIZE` for a quicker run.

In [ ]:
SAMPLE_SIZE = 200000

def fetch_socrata(dataset_id, limit, page_size=10000, **params):
    """Page through a DataSF dataset with $limit/$offset until `limit` rows or the end."""
    url = f'https://data.sf.gov/resource/{dataset_id}.json'
    rows = []
    while len(rows) < limit:
        page = {**params, '$limit': min(page_size, limit - len(rows)), '$offset': len(rows)}
        r = requests.get(url, params=page, timeout=120)
        r.raise_for_status()
        batch = r.json()
        rows.extend(batch)
        print(f'  fetched {len(rows)} rows...', end='\r', flush=True)
        if len(batch) < page['$limit']:
            break
    return rows

rows = fetch_socrata('tkzw-k3nq', SAMPLE_SIZE, page_size=25000, **{'$order': 'treeid'})
print(f'\n✅ Fetched {len(rows)} trees')
print()
for k, v in list(rows[0].items())[:16]:
    print(f'  {k:22s} {v}')

## 3. Create index with explicit mappings

In [ ]:
INDEX = 'sf_street_trees'

mapping = {'mappings': {'properties': {
    'tree_id':         {'type': 'keyword'},
    'scientific_name': {'type': 'keyword'},
    'common_name':     {'type': 'text', 'fields': {'keyword': {'type': 'keyword'}}},
    'legal_status':    {'type': 'keyword'},   # DPW Maintained, Significant Tree, Landmark tree, ...
    'caretaker':       {'type': 'keyword'},   # who looks after it
    'site_info':       {'type': 'keyword'},   # e.g. Sidewalk: Curb side : Cutout
    'plant_type':      {'type': 'keyword'},
    'planted_date':    {'type': 'date'},
    'dbh_inches':      {'type': 'float'},     # trunk diameter at breast height
    'plot_size':       {'type': 'keyword'},
    'address':         {'type': 'text'},
    'street':          {'type': 'keyword'},
    'neighborhood':    {'type': 'keyword'},
    'supervisor_district': {'type': 'keyword'},
    'zip':             {'type': 'keyword'},
    'location':        {'type': 'geo_point'},
}}}

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
    print(f'🗑️  Deleted existing index: {INDEX}')
es.indices.create(index=INDEX, body=mapping)
print(f'✅ Created index: {INDEX}')

## 4. Enrich and ingest
Species come as `"Scientific :: Common"` - we split them so you can aggregate on either. `description` holds the address (e.g. `167 Beaver St | Tree 1`).

In [ ]:
def geo_point(lat, lon):
    """Build a geo_point from lat/lon strings, skipping blanks and 0,0."""
    try:
        lat_f, lon_f = float(lat), float(lon)
    except (TypeError, ValueError):
        return None
    if lat_f == 0.0 or lon_f == 0.0:
        return None
    return {'lat': lat_f, 'lon': lon_f}

def to_float(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return None

def enrich(r):
    sci, _, common = (r.get('species') or '').partition('::')
    dbh = to_float(r.get('mapdbh'))
    doc = {
        'tree_id':         r.get('treeid'),
        'scientific_name': sci.strip(),
        'common_name':     common.strip(),
        'legal_status':    r.get('legalstatus'),
        'caretaker':       r.get('planter'),
        'site_info':       r.get('siteinfo'),
        'plant_type':      r.get('planttype'),
        'planted_date':    (r.get('planteddate') or '')[:10],
        'dbh_inches':      dbh if dbh and dbh < 400 else None,   # a few rows have impossible sizes
        'plot_size':       r.get('plotsize'),
        'address':         (r.get('description') or '').split('|')[0].strip(),
        'street':          r.get('onstreet'),
        'neighborhood':    r.get('analysis_neighborhood'),
        'supervisor_district': r.get('supervisordistrict'),
        'zip':             r.get('zip_code'),
        'location':        geo_point(r.get('latitude'), r.get('longitude')),
    }
    return {k: v for k, v in doc.items() if v not in (None, '')}

docs = [enrich(r) for r in rows]
success, errors = helpers.bulk(es, [{'_index': INDEX, '_source': d} for d in docs], raise_on_error=False)
es.indices.refresh(index=INDEX)
print(f'✅ Indexed {success} documents into {INDEX} ({len(errors)} errors)')

## 5. Verify - what grows on SF streets?

In [ ]:
resp = es.search(index=INDEX, body={'size': 0, 'track_total_hits': True, 'aggs': {
    'top_species':     {'terms': {'field': 'common_name.keyword', 'size': 10}},
    'by_neighborhood': {'terms': {'field': 'neighborhood', 'size': 8}},
    'by_status':       {'terms': {'field': 'legal_status', 'size': 8}},
}})
print(f"Total trees: {resp['hits']['total']['value']}\n")
print('Most common species:')
for b in resp['aggregations']['top_species']['buckets']:
    print(f"  {b['key']:32s} {b['doc_count']}")
print('\nLeafiest neighborhoods (by count):')
for b in resp['aggregations']['by_neighborhood']['buckets']:
    print(f"  {b['key']:32s} {b['doc_count']}")
print('\nLegal status:')
for b in resp['aggregations']['by_status']['buckets']:
    print(f"  {b['key']:32s} {b['doc_count']}")

In [ ]:
# The biggest trees within 1 km of the venue
resp = es.search(index=INDEX, body={'size': 5,
    'query': {'geo_distance': {'distance': '1km', 'location': {'lat': 37.7873, 'lon': -122.4005}}},
    'sort': [{'dbh_inches': {'order': 'desc', 'missing': '_last'}}],
    '_source': ['common_name', 'scientific_name', 'dbh_inches', 'address'],
})
print('Biggest street trees near 33 New Montgomery:')
for h in resp['hits']['hits']:
    s = h['_source']
    print(f"  {s.get('dbh_inches', '?'):>5} in  {s.get('common_name') or s.get('scientific_name')} - {s.get('address')}")

## 6. Going further with Mistral

- **Tree guide:** a Mistral agent that answers "what's the tree outside 167 Beaver St, and does it drop fruit?" by looking up the tree here and adding what it knows (or finds with web search).
- **Shade map + story:** aggregate canopy by neighborhood, then have a Mistral model write a short brief on where new plantings would help most.
- **Snap a leaf:** send a photo to a vision-capable Mistral model, guess the species, and find the nearest matching street trees.
- **Combine:** join with `Tree Maintenance` requests in [sf_311_cases](sf_311_cases.ipynb) to find blocks with repeated problems.

See [mistral_guide.md](mistral_guide.md) for calling Mistral directly, and [using_mistral_in_elasticsearch.md](using_mistral_in_elasticsearch.md) for semantic search.

---
## ✅ Index loaded
`sf_street_trees` is ready. Map it in Kibana, or ask "which neighborhood has the most London plane trees?"

**To refresh the data**, re-run cells 2-4. The index is dropped and recreated each time, so there are no duplicates.